# CapMuon vs. Plain Muon vs. AdamW on LLM RLVR (Multi-Epoch GRPO)

This self-contained notebook tests **CapMuon** ($O(d_{\text{in}})$ capped input-geometry preconditioning for Muon) against **Plain Muon** and **AdamW** on **Verifiable Reasoning RL (GRPO on Countdown)** using modern small open models (default: **`Qwen/Qwen3.5-0.8B`**, a 24-layer hybrid with 18 **Gated DeltaNet** linear-recurrent layers + 6 full-attention layers).

### Why CapMuon for LLM RLVR?
- **Exact mechanism**: Hidden layers exhibit extreme rank-1 input anisotropy ($\lambda_1 / \bar{\lambda} = 50\text{--}1000\times$) aligned with the batch mean direction $\mathbb{E}[x]$. Plain Muon orthogonalizes gradients equally in every input direction, including that dominant direction, which drives large coherent activation shifts under **non-stationary multi-epoch RL policy updates**.
- **CapMuon fix**: Tracks only the top-1 input eigenmode $(v_1, \lambda_1)$ per 2D linear layer via matrix-free $O(Bd)$ EMA power iteration and applies a one-sided cap $S = I - (1 - s_1) v_1 v_1^\top \preceq I$ with $s_1 = \min(1, (c\bar{\lambda}/\lambda_1)^p)$:
  $$\Delta W = -\eta \, \mathrm{polar}(Z S) \, S \, \sqrt{\max(1, R/C)}$$
  Zero $d \times d$ matrices, zero tail amplification ($S \preceq I$), and $<1\%$ wall-clock overhead.

### Recommended Colab Runtime
- **GPU**: `L4` (24 GB, ~2 Compute Units/hr) or `A100` (40/80 GB, ~7-12 Compute Units/hr).
- **Workflow**:
  1. Run **Cells 1-4** first (~2-3 minutes) to inspect the **exact input eigenspectrum** of your chosen model before spending Compute Units on training.
  2. Run **Cells 5-6** (~15-25 min on A100 / ~30-45 min on L4 for all 4 arms at 40 GRPO steps) to compare `AdamW`, `Plain Muon`, `CapMuon (p=0.25)`, and `CapMuon (p=0.50)` apples-to-apples from identical weights and prompt streams.


## 1. Environment Setup, GPU Check & Experiment Config


In [ ]:
# Install latest transformers (>=4.57 / 5.x required for Qwen3.5 Gated DeltaNet)
!pip install -qU "transformers>=4.57.0" accelerate matplotlib

# Optional (faster Gated DeltaNet kernels on A100/L4, skip if you want zero build wait):
# !pip install -q flash-linear-attention causal-conv1d

import gc
import json
import math
import os
import random
import re
import time
import ast
from collections import Counter
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

print(f"PyTorch: {torch.__version__} | Transformers: {transformers.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB | bf16: {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: No GPU detected! In Colab, go to Runtime -> Change runtime type -> L4 or A100.")

# ==============================================================================
# EXPERIMENT CONFIGURATION (edit here)
# ==============================================================================
CFG = {
    # Recommended 2026 targets (uncomment any one):
    "MODEL_ID": "Qwen/Qwen3.5-0.8B",                  # 18 Gated DeltaNet + 6 Attn (hybrid recurrent, fastest)
    # "MODEL_ID": "Qwen/Qwen3-0.6B",                  # Pure Transformer baseline (0.6B)
    # "MODEL_ID": "nvidia/Hymba-1.5B-Instruct",       # Parallel Mamba + Attention hybrid (1.5B)
    # "MODEL_ID": "nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16",  # Hybrid Mamba-2 + Attention (4B, use A100)
    # "MODEL_ID": "Qwen/Qwen2.5-1.5B-Instruct",       # Classic TinyZero Countdown baseline

    "SEED": 42,

    # GRPO schedule (40 steps x 4 arms is ~15-25 min on A100, ~30-45 min on L4)
    "NUM_TRAIN_STEPS": 40,
    "PROMPTS_PER_STEP": 8,       # Distinct Countdown puzzles per rollout step
    "GROUP_SIZE": 8,             # Rollouts per puzzle (64 total rollouts/step)
    "PPO_EPOCHS": 4,             # Multi-epoch GRPO updates per rollout batch (non-stationary regime!)
    "MICRO_BATCH": 4,            # Keep at 4 on L4 (24GB) for Qwen3.5's 248K vocab; 8-16 on A100
    "MAX_NEW_TOKENS": 224,
    "TEMPERATURE": 0.6,          # Calibrated for 0.8B arithmetic (0.9 adds too much digit noise)
    "TOP_P": 0.9,
    "CLIP_EPS": 0.2,
    "GRAD_CLIP": 1.0,

    # Calibrated post-training RLVR learning rates (verified on RTX 3080 Ti with Qwen3.5-0.8B!)
    # Note: momentum=0.85 prevents 40-step length drift; 1e-4 already destabilized Plain Muon by Step 5.
    "LR_ADAM": 5e-6,
    "LR_MUON": 5e-5,
    "MUON_MOMENTUM": 0.85,
    "WEIGHT_DECAY": 0.0,

    # Held-out validation
    "VAL_SIZE": 64,
    "EVAL_EVERY": 10,
    "EVAL_BATCH": 16,

    # Arms to run (remove or add stress-test arms like "muon_hi", "cap_p025_hi")
    "ARMS": [
        {"name": "adamw",    "type": "adamw",   "lr_adam": 5e-6, "lr_muon": 0.0,  "cap": False, "cap_p": 0.0},
        {"name": "muon",     "type": "capmuon", "lr_adam": 5e-6, "lr_muon": 5e-5, "cap": False, "cap_p": 0.0},
        {"name": "cap_p025", "type": "capmuon", "lr_adam": 5e-6, "lr_muon": 5e-5, "cap": True,  "cap_p": 0.25},
        {"name": "cap_p050", "type": "capmuon", "lr_adam": 5e-6, "lr_muon": 5e-5, "cap": True,  "cap_p": 0.50},
        # Optional high-LR stress-test arms (uncomment to test LR headroom!):
        # {"name": "muon_2x",     "type": "capmuon", "lr_adam": 5e-6, "lr_muon": 1e-4, "cap": False, "cap_p": 0.0},
        # {"name": "cap_p025_2x", "type": "capmuon", "lr_adam": 5e-6, "lr_muon": 1e-4, "cap": True,  "cap_p": 0.25},
    ],
}

print(f"\nTarget model: {CFG['MODEL_ID']}")
print(f"GRPO setup:   {CFG['NUM_TRAIN_STEPS']} steps | {CFG['PROMPTS_PER_STEP']}x{CFG['GROUP_SIZE']} rollouts/step | mu={CFG['PPO_EPOCHS']} PPO epochs")


## 2. Standalone PyTorch `CapMuon` Implementation ($O(Bd)$ Matrix-Free)

- `CapTracker` runs deterministic rank-1 EMA power iteration on $F = \mathbb{E}[x x^\top]$ without ever forming a $d \times d$ matrix.
- `CapMuon.attach_hooks(model)` automatically discovers all managed 2D `nn.Linear` hidden layers (Gated DeltaNet, Mamba, Attention, SwiGLU) while excluding tied embeddings, `lm_head`, and any frozen vision tower.
- `set_capture(True)` is enabled **only** on the forward pass of `ppo_epoch == 0, micro_idx == 0` (and disabled before `.backward()` so gradient checkpointing recomputation never double-counts activations).


In [ ]:
@torch.no_grad()
def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """5-step Quintic Newton-Schulz polar factor approximation."""
    assert G.ndim == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    work_dtype = torch.bfloat16 if (G.is_cuda and torch.cuda.is_bf16_supported()) else torch.float32
    X = G.to(work_dtype)
    tall = X.size(0) > X.size(1)
    if tall:
        X = X.T
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    if tall:
        X = X.T
    return X


class CapTracker:
    """Matrix-free O(k * B * d) streaming EMA power/subspace iteration for F = E[x x^T]."""

    def __init__(self, d: int, device: torch.device, rank: int = 1,
                 ema: float = 0.9, cap_c: float = 1.0, cap_p: float = 0.5):
        self.d = d
        self.rank = rank
        self.ema = ema
        self.cap_c = cap_c
        self.cap_p = cap_p
        self.steps = 0
        self.V = torch.zeros(d, rank, device=device, dtype=torch.float32)
        self.V[0, 0] = 1.0
        self.lam = torch.zeros(rank, device=device, dtype=torch.float32)
        self.tr_ema = torch.tensor(0.0, device=device, dtype=torch.float32)
        self.s = torch.ones(rank, device=device, dtype=torch.float32)
        self.top_ratio = 1.0

    @torch.no_grad()
    def update(self, x: torch.Tensor):
        X = x.detach().reshape(-1, self.d).float()
        B = X.size(0)
        if B == 0:
            return
        if self.steps == 0:
            mu = X.mean(dim=0)
            mn = mu.norm()
            if mn > 1e-12:
                self.V[:, 0] = mu / mn
            if self.rank > 1:
                g = torch.Generator(device=X.device).manual_seed(42)
                R = torch.randn(self.d, self.rank, device=X.device, generator=g)
                R[:, 0] = self.V[:, 0]
                Q, _ = torch.linalg.qr(R)
                self.V.copy_(Q)
            Y = X @ self.V
            self.lam.copy_((Y * Y).mean(dim=0))
            self.tr_ema.copy_((X * X).sum(dim=-1).mean())
            self.steps = 1
            return

        g = 0.0 if self.steps == 1 else self.ema
        Y = X @ self.V                           # [B, k]
        b_lam = (Y * Y).mean(dim=0)              # [k] exact Rayleigh quotients
        b_tr = (X * X).sum(dim=-1).mean()        # scalar exact trace
        GV = (X.T @ Y) / float(B)                # [d, k] exact F @ V

        prev_lam = self.lam.clone()
        self.lam.mul_(g).add_(b_lam, alpha=1.0 - g)
        self.tr_ema.mul_(g).add_(b_tr, alpha=1.0 - g)

        U = (g * prev_lam.unsqueeze(0)) * self.V + (1.0 - g) * GV
        if self.rank == 1:
            un = U[:, 0].norm()
            if un > 1e-12:
                self.V[:, 0] = U[:, 0] / un
        else:
            Q, _ = torch.linalg.qr(U)
            self.V.copy_(Q)

        mean_eig = self.tr_ema / float(self.d)
        thresh = self.cap_c * mean_eig
        if thresh > 1e-12:
            ratio = (thresh / self.lam.clamp_min(1e-12)).clamp_max(1.0)
            self.s.copy_(ratio.pow(self.cap_p))
            self.top_ratio = float((self.lam[0] / mean_eig).item())
        else:
            self.s.fill_(1.0)
            self.top_ratio = 1.0
        self.steps += 1


class CapMuon(torch.optim.Optimizer):
    """Muon with optional O(d_in) one-sided input-geometry capping (CapMuon)."""

    def __init__(self, params, lr: float = 1.5e-4, momentum: float = 0.95,
                 nesterov: bool = True, ns_steps: int = 5, weight_decay: float = 0.0,
                 cap: bool = True, cap_p: float = 0.5, cap_c: float = 1.0,
                 cap_ema: float = 0.9, cap_rank: int = 1):
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov,
                        ns_steps=ns_steps, weight_decay=weight_decay)
        super().__init__(params, defaults)
        self.cap = cap
        self.cap_p = cap_p
        self.cap_c = cap_c
        self.cap_ema = cap_ema
        self.cap_rank = cap_rank
        self.trackers: dict[torch.nn.Parameter, CapTracker] = {}
        self._handles = []
        self._capture_now = False

    def attach_hooks(self, model: nn.Module) -> int:
        managed = {p for g in self.param_groups for p in g["params"]}
        attached = 0
        for _name, mod in model.named_modules():
            if isinstance(mod, nn.Linear) and mod.weight in managed:
                d_in = mod.weight.shape[1]
                tracker = CapTracker(
                    d=d_in, device=mod.weight.device, rank=self.cap_rank,
                    ema=self.cap_ema, cap_c=self.cap_c, cap_p=self.cap_p,
                )
                self.trackers[mod.weight] = tracker

                def _make_hook(tr):
                    def _hook(_mod, inputs):
                        if self.cap and self._capture_now and torch.is_grad_enabled():
                            if inputs and torch.is_tensor(inputs[0]):
                                tr.update(inputs[0])
                    return _hook

                self._handles.append(mod.register_forward_pre_hook(_make_hook(tracker)))
                attached += 1
        return attached

    def remove_hooks(self):
        for h in self._handles:
            h.remove()
        self._handles.clear()

    def set_capture(self, flag: bool):
        self._capture_now = flag

    def get_cap_summary(self) -> dict:
        if not self.trackers:
            return {"max_ratio": 1.0, "med_ratio": 1.0, "min_s": 1.0}
        ratios = [tr.top_ratio for tr in self.trackers.values()]
        s_vals = [float(tr.s[0].item()) for tr in self.trackers.values()]
        return {
            "max_ratio": float(max(ratios)),
            "med_ratio": float(np.median(ratios)),
            "min_s": float(min(s_vals)),
        }

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr = group["lr"]
            mom = group["momentum"]
            nesterov = group["nesterov"]
            ns_steps = group["ns_steps"]
            wd = group["weight_decay"]
            for p in group["params"]:
                g = p.grad
                if g is None:
                    continue
                state = self.state[p]
                if "buf" not in state:
                    state["buf"] = torch.zeros_like(g)
                buf = state["buf"]
                buf.mul_(mom).add_(g)
                Z = g.add(buf, alpha=mom) if nesterov else buf

                tr = self.trackers.get(p, None) if self.cap else None
                apply_cap = (tr is not None) and (tr.steps >= 2) and bool((tr.s < 1.0).any())

                if apply_cap:
                    V = tr.V.to(Z.dtype)
                    one_minus_s = (1.0 - tr.s).to(Z.dtype)
                    ZV = Z @ V
                    Z_in = Z - (ZV * one_minus_s.unsqueeze(0)) @ V.T
                else:
                    Z_in = Z

                Q = zeropower_via_newtonschulz5(Z_in, steps=ns_steps)

                if apply_cap:
                    V = tr.V.to(Q.dtype)
                    one_minus_s = (1.0 - tr.s).to(Q.dtype)
                    QV = Q @ V
                    Q = Q - (QV * one_minus_s.unsqueeze(0)) @ V.T

                R, C = p.shape
                scale = math.sqrt(max(1.0, R / C))
                if wd > 0.0:
                    p.mul_(1.0 - lr * wd)
                p.add_(Q.to(p.dtype), alpha=-lr * scale)
        return loss


def split_muon_and_adam_params(model: nn.Module):
    """
    Split trainable parameters into:
      - muon_params: 2D nn.Linear hidden weights (excluding embeddings, lm_head, vision)
      - adam_params: embeddings, lm_head, 1D norms/biases/conv1d/SSM scalars
    Safely deduplicates tied weights (e.g., tie_word_embeddings=True in Qwen3.5).
    """
    excluded_ids = set()
    for name, mod in model.named_modules():
        lname = name.lower()
        if isinstance(mod, nn.Embedding) or any(k in lname for k in ("embed", "lm_head", "wte", "wpe", "visual", "vision")):
            for p in mod.parameters(recurse=False):
                excluded_ids.add(id(p))

    muon_params = []
    muon_ids = set()
    for name, mod in model.named_modules():
        lname = name.lower()
        if isinstance(mod, nn.Linear) and not any(k in lname for k in ("embed", "lm_head", "visual", "vision")):
            w = mod.weight
            if w.requires_grad and w.ndim == 2 and id(w) not in excluded_ids and id(w) not in muon_ids:
                muon_params.append(w)
                muon_ids.add(id(w))

    adam_params = []
    seen = set(muon_ids)
    for p in model.parameters():
        if p.requires_grad and id(p) not in seen:
            adam_params.append(p)
            seen.add(id(p))

    return muon_params, adam_params


def build_arm_optimizers(model: nn.Module, arm: dict, weight_decay: float = 0.0, muon_momentum: float = 0.85):
    """Build optimizers for an experiment arm and attach CapMuon hooks if enabled."""
    muon_params, adam_params = split_muon_and_adam_params(model)
    if arm["type"] == "adamw":
        all_trainable = muon_params + adam_params
        opt = torch.optim.AdamW(all_trainable, lr=arm["lr_adam"], betas=(0.9, 0.95), weight_decay=weight_decay)
        return [opt], None, len(muon_params), 0

    opts = []
    if adam_params:
        opts.append(torch.optim.AdamW(adam_params, lr=arm["lr_adam"], betas=(0.9, 0.95), weight_decay=weight_decay))
    cap_opt = CapMuon(
        muon_params,
        lr=arm["lr_muon"],
        momentum=muon_momentum,
        nesterov=True,
        ns_steps=5,
        weight_decay=weight_decay,
        cap=arm["cap"],
        cap_p=arm["cap_p"],
        cap_c=1.0,
        cap_ema=0.9,
        cap_rank=1,
    )
    attached = cap_opt.attach_hooks(model) if arm["cap"] else 0
    opts.append(cap_opt)
    return opts, cap_opt, len(muon_params), attached

print("CapMuon & parameter splitter ready.")


## 3. Verifiable Reasoning Task: Countdown (3- & 4-Number Mix)

Every puzzle is generated by constructing a valid integer expression tree, guaranteeing **100% solvability** with exact rational verification (`fractions.Fraction`, zero `eval()`).

- **Reward structure** (strict binary verifiable reward — prevents small models from reward-hacking short wrong answers):
  - `1.0` (**Solved**): Valid arithmetic expression inside `<answer>...</answer>` using the exact multiset of given numbers and evaluating exactly to `target`.
  - `0.0`: Anything else.
- **Primary metric reported**: Exact **Solve Rate** (`reward == 1.0`).


In [ ]:
def generate_countdown_puzzle(rng: random.Random, n_nums: int = None) -> dict:
    """Generate a guaranteed-solvable Countdown puzzle with 3 or 4 numbers."""
    if n_nums is None:
        n_nums = 3 if rng.random() < 0.5 else 4
    hi = 12 if n_nums == 3 else 18

    for _ in range(500):
        nums = [rng.randint(1, hi) for _ in range(n_nums)]
        items = [(Fraction(x), str(x)) for x in nums]
        rng.shuffle(items)
        ok = True
        while len(items) > 1:
            i = rng.randrange(len(items))
            a_val, a_expr = items.pop(i)
            j = rng.randrange(len(items))
            b_val, b_expr = items.pop(j)
            ops = ["+", "-", "*"]
            if b_val != 0 and a_val % b_val == 0:
                ops.append("/")
            op = rng.choice(ops)
            if op == "+":
                val = a_val + b_val
            elif op == "-":
                if a_val < b_val:
                    a_val, b_val = b_val, a_val
                    a_expr, b_expr = b_expr, a_expr
                val = a_val - b_val
            elif op == "*":
                val = a_val * b_val
            else:
                val = a_val / b_val
            if val <= 0 or val > 250 or val.denominator != 1:
                ok = False
                break
            items.append((val, f"({a_expr} {op} {b_expr})"))
        if ok:
            target = int(items[0][0])
            if 10 <= target <= 120:
                shuffled = list(nums)
                rng.shuffle(shuffled)
                return {"nums": shuffled, "target": target, "ref_expr": items[0][1]}
    return {"nums": [3, 5, 8], "target": 37, "ref_expr": "((8 * 5) - 3)"}


def make_dataset(n: int, seed: int) -> list[dict]:
    rng = random.Random(seed)
    return [generate_countdown_puzzle(rng) for _ in range(n)]


SYSTEM_PROMPT = (
    "You are a concise arithmetic solver. Using EVERY given number exactly once "
    "with +, -, *, / and parentheses, form an expression equal to the target. "
    "Show brief arithmetic checks inside <reasoning>...</reasoning> and output ONLY "
    "the final expression inside <answer>...</answer>."
)


def format_countdown_prompt(tokenizer, puzzle: dict) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Numbers: [3, 5, 8]\nTarget: 37"},
        {"role": "assistant", "content": "<reasoning>Try 8 * 5 = 40. Then 40 - 3 = 37. Matches 37.</reasoning>\n<answer>(8 * 5) - 3</answer>"},
        {"role": "user", "content": "Numbers: [2, 6, 9, 4]\nTarget: 46"},
        {"role": "assistant", "content": "<reasoning>Try 9 * 6 = 54. Remaining [2, 4] give 4 * 2 = 8. 54 - 8 = 46. Matches 46.</reasoning>\n<answer>(9 * 6) - (4 * 2)</answer>"},
        {"role": "user", "content": f"Numbers: {puzzle['nums']}\nTarget: {puzzle['target']}"},
    ]
    if getattr(tokenizer, "chat_template", None):
        try:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
        except TypeError:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
    return (
        f"{SYSTEM_PROMPT}\n\n"
        "User: Numbers: [3, 5, 8]\nTarget: 37\n"
        "Assistant: <reasoning>Try 8 * 5 = 40. Then 40 - 3 = 37. Matches 37.</reasoning>\n<answer>(8 * 5) - 3</answer>\n\n"
        f"User: Numbers: {puzzle['nums']}\nTarget: {puzzle['target']}\nAssistant: "
    )


_ANSWER_RE = re.compile(r"<answer>(.*?)</answer>", re.IGNORECASE | re.DOTALL)


def _safe_eval_ast(node, used_nums: list[int]) -> Fraction:
    if isinstance(node, ast.Expression):
        return _safe_eval_ast(node.body, used_nums)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        if isinstance(node.value, float) and not node.value.is_integer():
            raise ValueError("Non-integer literal")
        iv = int(node.value)
        used_nums.append(iv)
        return Fraction(iv)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.UAdd):
        return _safe_eval_ast(node.operand, used_nums)
    if isinstance(node, ast.BinOp):
        left = _safe_eval_ast(node.left, used_nums)
        right = _safe_eval_ast(node.right, used_nums)
        if isinstance(node.op, ast.Add):
            return left + right
        if isinstance(node.op, ast.Sub):
            return left - right
        if isinstance(node.op, ast.Mult):
            return left * right
        if isinstance(node.op, ast.Div):
            if right == 0:
                raise ZeroDivisionError()
            return left / right
    raise ValueError("Unsupported AST node")


def score_countdown(response: str, nums: list[int], target: int) -> tuple[float, bool]:
    """Return (reward, is_solved) using exact rational arithmetic."""
    matches = _ANSWER_RE.findall(response)
    if not matches:
        return 0.0, False
    expr = matches[-1].strip()
    if "=" in expr:
        expr = expr.split("=")[0].strip()
    expr = expr.replace("×", "*").replace("÷", "/").replace("−", "-")
    if not expr or len(expr) > 80:
        return 0.0, False
    try:
        tree = ast.parse(expr, mode="eval")
        used = []
        val = _safe_eval_ast(tree, used)
    except Exception:
        return 0.0, False

    if Counter(used) == Counter(nums) and val == Fraction(target):
        return 1.0, True
    return 0.0, False


# Quick self-test of generator + verifier
_val_preview = make_dataset(5, seed=CFG["SEED"] + 999)
for ex in _val_preview:
    r, ok = score_countdown(f"<think>ok</think><answer>{ex['ref_expr']}</answer>", ex["nums"], ex["target"])
    assert ok and r == 1.0, f"Self-test failed on {ex}"
print("Countdown generator & exact verifier self-test passed! Sample puzzles:")
for ex in _val_preview[:3]:
    print(f"  nums={ex['nums']} -> target={ex['target']}  (e.g. {ex['ref_expr']})")


## 4. Load Model & Run 2-Minute Input Eigenspectrum Check

Before spending Compute Units on RLVR training, let's verify that `MODEL_ID` exhibits the exact **rank-1 input eigenspectrum cliff** ($\lambda_1 / \bar{\lambda} \gg 1$, $|\cos(v_1, \mathbb{E}[x])| \approx 1$) across its **Gated DeltaNet / Mamba / Attention / SwiGLU** layers.

> **Note for `Qwen/Qwen3.5-0.8B`**: Qwen3.5 ships as a unified multimodal checkpoint (`Qwen3_5ForConditionalGeneration`). The loader below automatically freezes the unused vision encoder so 100% of VRAM and optimizer compute goes to the text backbone.


In [ ]:
def load_model_and_tokenizer(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    cfg = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    is_vlm = hasattr(cfg, "vision_config") or "ConditionalGeneration" in str(getattr(cfg, "architectures", []))

    load_kwargs = {"trust_remote_code": True}
    # Handle transformers 4.x (torch_dtype) vs 5.x (dtype)
    try:
        if is_vlm and AutoModelForImageTextToText is not None:
            model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=DTYPE, **load_kwargs)
        else:
            model = AutoModelForCausalLM.from_pretrained(model_id, dtype=DTYPE, **load_kwargs)
    except TypeError:
        if is_vlm and AutoModelForImageTextToText is not None:
            model = AutoModelForImageTextToText.from_pretrained(model_id, torch_dtype=DTYPE, **load_kwargs)
        else:
            model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=DTYPE, **load_kwargs)

    # Freeze any vision encoder parameters so text-only RLVR pays zero vision overhead
    frozen_vis = 0
    for name, p in model.named_parameters():
        lname = name.lower()
        if "visual" in lname or "vision" in lname:
            p.requires_grad = False
            frozen_vis += p.numel()

    model.to(DEVICE)
    model.config.use_cache = False
    if hasattr(model.config, "text_config") and model.config.text_config is not None:
        model.config.text_config.use_cache = False

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    muon_p, adam_p = split_muon_and_adam_params(model)
    print(f"Loaded {model_id} ({model.__class__.__name__})")
    print(f"  Trainable params: {n_train / 1e6:.1f}M | Frozen vision: {frozen_vis / 1e6:.1f}M")
    print(f"  Muon/CapMuon 2D matrices: {len(muon_p)} ({sum(p.numel() for p in muon_p)/1e6:.1f}M params)")
    print(f"  AdamW (embed/head/norms/1D): {len(adam_p)} ({sum(p.numel() for p in adam_p)/1e6:.1f}M params)")
    return model, tokenizer


def find_decoder_layers(model: nn.Module):
    """Locate the list of transformer/hybrid decoder blocks across HF architectures."""
    candidates = [
        ("model", "language_model", "layers"),
        ("model", "layers"),
        ("language_model", "model", "layers"),
        ("transformer", "h"),
        ("backbone", "layers"),
    ]
    for path in candidates:
        obj = model
        ok = True
        for attr in path:
            if hasattr(obj, attr):
                obj = getattr(obj, attr)
            else:
                ok = False
                break
        if ok and isinstance(obj, (nn.ModuleList, list)) and len(obj) > 0:
            return obj
    return None


@torch.no_grad()
def inspect_input_eigenspectra(model: nn.Module, tokenizer, n_prompts: int = 16):
    """Exact F = X^T X / N eigenspectrum across early, middle, and late decoder blocks."""
    layers = find_decoder_layers(model)
    if layers is None:
        print("Could not locate decoder layers automatically; skipping spectrum plot.")
        return {}

    L = len(layers)
    probe_indices = sorted(set([0, L // 4, L // 2, (3 * L) // 4, L - 1]))
    # Pick unique linear projections inside those layers (both linear_attn / self_attn / mlp)
    captured = {}
    handles = []

    for idx in probe_indices:
        blk = layers[idx]
        for sub_name, mod in blk.named_modules():
            if isinstance(mod, nn.Linear) and mod.weight.requires_grad:
                key = f"L{idx}.{sub_name}"
                captured[key] = []

                def _make_hook(k):
                    def _hook(_m, inp):
                        if inp and torch.is_tensor(inp[0]):
                            # Flatten batch*seq -> [N, d_in] in float32 on CPU to save VRAM
                            captured[k].append(inp[0].detach().reshape(-1, inp[0].shape[-1]).float().cpu())
                    return _hook

                handles.append(mod.register_forward_pre_hook(_make_hook(key)))

    puzzles = make_dataset(n_prompts, seed=CFG["SEED"] + 77)
    prompts = [format_countdown_prompt(tokenizer, p) for p in puzzles]
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)

    model.eval()
    _ = model(**enc, use_cache=False)
    for h in handles:
        h.remove()

    # Keep only non-pad positions
    mask = enc["attention_mask"].reshape(-1).bool().cpu()
    stats = {}
    print(f"\n{'Layer':<36} {'Shape':>12} {'lam1/mean':>10} {'lam1/lam2':>10} {'%E1':>7} {'|cos(v1,E[x])|':>15}")
    print("-" * 96)

    for key, chunks in captured.items():
        if not chunks:
            continue
        X = torch.cat(chunks, dim=0)
        if X.shape[0] == mask.shape[0]:
            X = X[mask]
        N, d = X.shape
        if N < 16 or d > 4096:
            # Cap extremely wide projections for speed if needed (e.g. >4096)
            if d > 4096:
                continue
        F_mat = (X.T @ X) / float(N)
        evals, evecs = torch.linalg.eigh(F_mat)
        evals = evals.flip(0).clamp_min(0.0)
        v1 = evecs[:, -1]
        mean_eig = evals.mean().item()
        l1 = evals[0].item()
        l2 = max(evals[1].item(), 1e-12)
        mu = X.mean(dim=0)
        cos_mu = abs(float((v1 @ mu).item()) / (mu.norm().item() + 1e-12))
        pct_e1 = 100.0 * l1 / max(evals.sum().item(), 1e-12)
        ratio = l1 / max(mean_eig, 1e-12)
        cliff = l1 / l2
        stats[key] = {
            "d_in": d,
            "ratio": ratio,
            "cliff": cliff,
            "pct_e1": pct_e1,
            "cos_mu": cos_mu,
            "top32": (evals[:32] / max(mean_eig, 1e-12)).numpy(),
        }

    # Print top 18 most spiked layers + representative summary
    ranked = sorted(stats.items(), key=lambda kv: kv[1]["ratio"], reverse=True)
    for key, st in ranked[:18]:
        shp = f"(*,{st['d_in']})"
        print(f"{key:<36} {shp:>12} {st['ratio']:>9.1f}x {st['cliff']:>9.1f}x {st['pct_e1']:>6.1f}% {st['cos_mu']:>15.4f}")

    # Plot dark-mode 2-panel eigenspectrum diagnostic
    if ranked:
        plt.style.use("dark_background")
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.8), facecolor="#0d1117")
        for ax in (ax1, ax2):
            ax.set_facecolor("#0d1117")
            ax.grid(alpha=0.18, linestyle="--")

        palette = ["#ff6b6b", "#ffa94d", "#4dabf7", "#69db7c", "#da77f2", "#ffd43b"]
        for i, (key, st) in enumerate(ranked[:6]):
            ax1.plot(range(1, len(st["top32"]) + 1), st["top32"], marker="o", ms=3.5, lw=2.0,
                     color=palette[i % len(palette)], label=f"{key} ({st['ratio']:.0f}x)")
        ax1.axhline(1.0, color="#8b949e", ls=":", lw=1.2, label="Isotropic mean (1.0x)")
        ax1.set_yscale("log")
        ax1.set_xlabel("Eigenvalue Rank i")
        ax1.set_ylabel("Normalized Eigenvalue  λᵢ / mean(λ)")
        ax1.set_title(f"{CFG['MODEL_ID']}: Top-32 Input Eigenspectra", fontweight="bold")
        ax1.legend(fontsize=8, frameon=False)

        top12 = ranked[:12][::-1]
        names = [k for k, _ in top12]
        vals = [v["ratio"] for _, v in top12]
        colors = ["#ff6b6b" if ("linear_attn" in n or "mamba" in n or "ssm" in n)
                  else ("#4dabf7" if "attn" in n else "#69db7c") for n in names]
        ax2.barh(names, vals, color=colors, edgecolor="#21262d")
        ax2.set_xlabel("Top Input Spike  λ₁ / mean(λ)")
        ax2.set_title("Most Anisotropic Linear Layers\n(Red=Recurrent/GDN, Blue=Attn, Green=MLP)", fontweight="bold")
        ax2.tick_params(axis="y", labelsize=8.5)

        plt.tight_layout()
        plt.savefig("model_eigenspectrum.png", dpi=160, facecolor="#0d1117")
        plt.show()
    return stats


model, tokenizer = load_model_and_tokenizer(CFG["MODEL_ID"])
spec_stats = inspect_input_eigenspectra(model, tokenizer, n_prompts=16)
torch.cuda.empty_cache()


## 5. Multi-Epoch GRPO Trainer ($\mu = 4$ PPO Epochs per Rollout Batch)

Key implementation details for exact, OOM-free RLVR benchmarking:
1. **Bit-identical initialization**: Snapshots all trainable weights to CPU once (`~1.5 GB` for `0.8B` in `bf16`) and restores them before every arm so `AdamW`, `Plain Muon`, `CapMuon (p=0.25)`, and `CapMuon (p=0.50)` start from the exact same model weights and see the exact same prompt stream.
2. **Memory-safe log-probs for 248K vocab**: Slices `logits` strictly to the generated completion tokens (`[:, prompt_len - 1 : -1, :]`) and uses fused `F.cross_entropy(..., reduction="none")` in `float32` so we never allocate a full `[B, T, 248320]` `log_softmax` tensor.
3. **Exact `CapTracker` cadence**: Enables `cap_opt.set_capture(True)` ONLY around the forward pass of `ppo_epoch == 0, micro_idx == 0` and sets it back to `False` immediately before `.backward()`, avoiding duplicate updates under gradient checkpointing.
4. **Honest stability diagnostics**: Logs held-out `val_acc`, `train_acc`, `train_reward`, multi-epoch `approx_kl` (unbiased Schulman k3 estimator), `clipfrac`, mean response length, and CapMuon `max_ratio` / `min_s`.


In [ ]:
# Snapshot initial trainable weights once on CPU so every arm starts identically
INIT_TRAINABLE_STATE = {
    name: p.detach().cpu().clone()
    for name, p in model.named_parameters()
    if p.requires_grad
}
VAL_SET = make_dataset(CFG["VAL_SIZE"], seed=CFG["SEED"] + 10000)
TRAIN_STREAM = make_dataset(CFG["NUM_TRAIN_STEPS"] * CFG["PROMPTS_PER_STEP"], seed=CFG["SEED"])
print(f"Snapshotted {len(INIT_TRAINABLE_STATE)} trainable tensors on CPU.")
print(f"Prepared {len(TRAIN_STREAM)} train puzzles and {len(VAL_SET)} fixed held-out val puzzles.")


def reset_model_to_init(model: nn.Module):
    with torch.no_grad():
        for name, p in model.named_parameters():
            if name in INIT_TRAINABLE_STATE:
                p.copy_(INIT_TRAINABLE_STATE[name].to(device=p.device, dtype=p.dtype))


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_completion_logprobs(model: nn.Module, seq_ids: torch.Tensor, attn_mask: torch.Tensor,
                                prompt_len: int) -> torch.Tensor:
    """
    Exact per-token log-probabilities strictly on completion positions [prompt_len .. seq_len-1].
    Uses fused F.cross_entropy on sliced logits to stay lean even with Qwen3.5's 248K vocab.
    """
    out = model(input_ids=seq_ids, attention_mask=attn_mask, use_cache=False)
    comp_logits = out.logits[:, prompt_len - 1 : -1, :].float()
    comp_targets = seq_ids[:, prompt_len:]
    B_m, T_c, V = comp_logits.shape
    nll = F.cross_entropy(
        comp_logits.reshape(B_m * T_c, V),
        comp_targets.reshape(B_m * T_c),
        reduction="none",
    ).view(B_m, T_c)
    return -nll


def _set_grad_ckpt(model: nn.Module, enabled: bool):
    if enabled:
        if hasattr(model, "activation_checkpointing_enable"):
            model.activation_checkpointing_enable()
        elif hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
    else:
        if hasattr(model, "activation_checkpointing_disable"):
            model.activation_checkpointing_disable()
        elif hasattr(model, "gradient_checkpointing_disable"):
            model.gradient_checkpointing_disable()


@torch.no_grad()
def evaluate_model(model: nn.Module, tokenizer, val_set: list[dict], batch_size: int = 16) -> dict:
    model.eval()
    _set_grad_ckpt(model, False)
    solved = 0
    total_r = 0.0
    total_len = 0
    for i in range(0, len(val_set), batch_size):
        batch = val_set[i : i + batch_size]
        prompts = [format_countdown_prompt(tokenizer, p) for p in batch]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
        prompt_len = enc["input_ids"].shape[1]
        gen = model.generate(
            **enc,
            max_new_tokens=CFG["MAX_NEW_TOKENS"],
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
        comp_ids = gen[:, prompt_len:]
        texts = tokenizer.batch_decode(comp_ids, skip_special_tokens=True)
        for txt, puz, ids in zip(texts, batch, comp_ids):
            r, ok = score_countdown(txt, puz["nums"], puz["target"])
            total_r += r
            solved += int(ok)
            total_len += int((ids != tokenizer.pad_token_id).sum().item())
    n = len(val_set)
    return {"val_acc": solved / n, "val_reward": total_r / n, "val_len": total_len / n}


def run_grpo_arm(model: nn.Module, tokenizer, arm: dict) -> dict:
    """Run one complete GRPO arm from identical initial weights and return full history."""
    print("\n" + "=" * 78)
    print(f"STARTING ARM: {arm['name']} | type={arm['type']} lr_adam={arm['lr_adam']} "
          f"lr_muon={arm['lr_muon']} cap={arm['cap']} cap_p={arm['cap_p']}")
    print("=" * 78)

    reset_model_to_init(model)
    set_all_seeds(CFG["SEED"])
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    opts, cap_opt, n_muon_mats, n_hooked = build_arm_optimizers(
        model, arm, weight_decay=CFG["WEIGHT_DECAY"], muon_momentum=CFG.get("MUON_MOMENTUM", 0.85)
    )
    print(f"Optimizers built | 2D matrices={n_muon_mats} | CapMuon hooks attached={n_hooked}")
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    history = {
        "arm": arm,
        "steps": [],
        "train_acc": [],
        "train_reward": [],
        "approx_kl": [],
        "clipfrac": [],
        "resp_len": [],
        "top_ratio": [],
        "min_s": [],
        "val_steps": [],
        "val_acc": [],
        "val_reward": [],
        "wall_sec": 0.0,
    }

    # Step 0 baseline validation
    t_start = time.time()
    val0 = evaluate_model(model, tokenizer, VAL_SET, batch_size=CFG["EVAL_BATCH"])
    history["val_steps"].append(0)
    history["val_acc"].append(val0["val_acc"])
    history["val_reward"].append(val0["val_reward"])
    print(f"[Step 000] INIT VAL | val_acc={val0['val_acc']*100:5.1f}% | val_r={val0['val_reward']:.3f} | len={val0['val_len']:.1f}")

    P = CFG["PROMPTS_PER_STEP"]
    G = CFG["GROUP_SIZE"]
    B = P * G
    MB = CFG["MICRO_BATCH"]
    eos_id = tokenizer.eos_token_id
    pad_id = tokenizer.pad_token_id

    for step in range(1, CFG["NUM_TRAIN_STEPS"] + 1):
        step_t0 = time.time()
        batch_puzzles = TRAIN_STREAM[(step - 1) * P : step * P]
        repeated_puzzles = [puz for puz in batch_puzzles for _ in range(G)]
        prompts = [format_countdown_prompt(tokenizer, puz) for puz in repeated_puzzles]

        # ------------------------------------------------------------------
        # 1. Rollout generation (eval mode, KV cache enabled, no_grad)
        # ------------------------------------------------------------------
        model.eval()
        _set_grad_ckpt(model, False)
        if cap_opt is not None:
            cap_opt.set_capture(False)

        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
        prompt_len = enc["input_ids"].shape[1]

        with torch.no_grad():
            seq_ids = model.generate(
                **enc,
                max_new_tokens=CFG["MAX_NEW_TOKENS"],
                do_sample=True,
                temperature=CFG["TEMPERATURE"],
                top_p=CFG["TOP_P"],
                pad_token_id=pad_id,
                eos_token_id=eos_id,
                use_cache=True,
            )

        comp_ids = seq_ids[:, prompt_len:]
        T_c = comp_ids.shape[1]
        if T_c == 0:
            continue

        # Build exact completion mask: 1 up to and including first EOS, 0 afterwards / on pad
        is_eos = (comp_ids == eos_id)
        eos_cumsum = is_eos.cumsum(dim=1)
        comp_mask = ((eos_cumsum == 0) | (is_eos & (eos_cumsum == 1))).float()
        if pad_id != eos_id:
            comp_mask = comp_mask * (comp_ids != pad_id).float()

        full_attn = torch.cat([enc["attention_mask"], comp_mask.long()], dim=1)

        # Verify rewards & compute exact GRPO group-normalized advantages
        texts = tokenizer.batch_decode(comp_ids, skip_special_tokens=True)
        rewards_list = []
        solved_list = []
        for txt, puz in zip(texts, repeated_puzzles):
            r, ok = score_countdown(txt, puz["nums"], puz["target"])
            rewards_list.append(r)
            solved_list.append(float(ok))

        rewards = torch.tensor(rewards_list, device=DEVICE, dtype=torch.float32).view(P, G)
        r_mean = rewards.mean(dim=1, keepdim=True)
        r_std = rewards.std(dim=1, keepdim=True)
        adv = torch.where(r_std > 1e-6, (rewards - r_mean) / (r_std + 1e-6), torch.zeros_like(rewards))
        adv = adv.reshape(B)

        train_acc = float(np.mean(solved_list))
        train_reward = float(rewards.mean().item())
        mean_resp_len = float(comp_mask.sum(dim=1).mean().item())

        # Skip optimizer update if all groups had zero variance (nothing to learn from this batch)
        if adv.abs().max().item() < 1e-6:
            kl_val, clip_val = 0.0, 0.0
        else:
            # --------------------------------------------------------------
            # 2. Cache rollout old_logprobs (microbatched, no_grad)
            # --------------------------------------------------------------
            old_lp_chunks = []
            with torch.no_grad():
                for start in range(0, B, MB):
                    end = min(start + MB, B)
                    lp = compute_completion_logprobs(
                        model, seq_ids[start:end], full_attn[start:end], prompt_len
                    )
                    old_lp_chunks.append(lp.detach())
            old_logprobs = torch.cat(old_lp_chunks, dim=0)

            # --------------------------------------------------------------
            # 3. Multi-Epoch Clipped PPO-GRPO Updates (mu = PPO_EPOCHS)
            # --------------------------------------------------------------
            model.train()
            _set_grad_ckpt(model, True)

            kl_accum = 0.0
            clip_accum = 0.0
            tok_accum = 0.0
            total_valid_tokens = comp_mask.sum().clamp_min(1.0)

            for ppo_ep in range(CFG["PPO_EPOCHS"]):
                for opt in opts:
                    opt.zero_grad(set_to_none=True)

                perm = torch.randperm(B, device=DEVICE)
                for mb_idx, start in enumerate(range(0, B, MB)):
                    idx = perm[start : start + MB]
                    mb_seq = seq_ids[idx]
                    mb_attn = full_attn[idx]
                    mb_mask = comp_mask[idx]
                    mb_old_lp = old_logprobs[idx]
                    mb_adv = adv[idx]

                    mb_toks = mb_mask.sum()
                    if mb_toks.item() < 1.0:
                        continue

                    # Capture input eigendirections ONLY once per rollout batch (epoch 0, micro 0)
                    capture_now = (cap_opt is not None) and arm["cap"] and (ppo_ep == 0) and (mb_idx == 0)
                    if cap_opt is not None:
                        cap_opt.set_capture(capture_now)

                    new_lp = compute_completion_logprobs(model, mb_seq, mb_attn, prompt_len)

                    # Disable capture BEFORE backward() so grad-checkpoint recompute never double-counts!
                    if cap_opt is not None:
                        cap_opt.set_capture(False)

                    log_ratio = (new_lp - mb_old_lp).clamp(-10.0, 10.0)
                    ratio = torch.exp(log_ratio)
                    surr1 = ratio * mb_adv.unsqueeze(1)
                    surr2 = ratio.clamp(1.0 - CFG["CLIP_EPS"], 1.0 + CFG["CLIP_EPS"]) * mb_adv.unsqueeze(1)
                    # Normalize by total batch tokens for exact gradient accumulation across microbatches
                    loss = -(torch.minimum(surr1, surr2) * mb_mask).sum() / total_valid_tokens
                    loss.backward()

                    with torch.no_grad():
                        # Unbiased k3 KL estimator: E[(r - 1) - log r] >= 0
                        kl_mb = (((ratio - 1.0) - log_ratio) * mb_mask).sum().item()
                        clip_mb = (((ratio - 1.0).abs() > CFG["CLIP_EPS"]).float() * mb_mask).sum().item()
                        kl_accum += kl_mb
                        clip_accum += clip_mb
                        tok_accum += mb_toks.item()

                torch.nn.utils.clip_grad_norm_(trainable_params, CFG["GRAD_CLIP"])
                for opt in opts:
                    opt.step()

            kl_val = kl_accum / max(tok_accum, 1.0)
            clip_val = clip_accum / max(tok_accum, 1.0)

        cap_stats = cap_opt.get_cap_summary() if (cap_opt is not None and arm["cap"]) else {"max_ratio": 1.0, "min_s": 1.0}

        history["steps"].append(step)
        history["train_acc"].append(train_acc)
        history["train_reward"].append(train_reward)
        history["approx_kl"].append(kl_val)
        history["clipfrac"].append(clip_val)
        history["resp_len"].append(mean_resp_len)
        history["top_ratio"].append(cap_stats["max_ratio"])
        history["min_s"].append(cap_stats["min_s"])

        dt = time.time() - step_t0
        if step == 1 or step % 5 == 0 or step == CFG["NUM_TRAIN_STEPS"]:
            cap_str = f" | max_lam={cap_stats['max_ratio']:6.1f}x min_s={cap_stats['min_s']:.2f}" if arm["cap"] else ""
            print(
                f"[Step {step:03d}/{CFG['NUM_TRAIN_STEPS']}] "
                f"acc={train_acc*100:5.1f}% r={train_reward:.3f} | "
                f"kl={kl_val:.4f} clip={clip_val*100:4.1f}% | len={mean_resp_len:5.1f} | {dt:.1f}s{cap_str}"
            )

        if step % CFG["EVAL_EVERY"] == 0 or step == CFG["NUM_TRAIN_STEPS"]:
            val_res = evaluate_model(model, tokenizer, VAL_SET, batch_size=CFG["EVAL_BATCH"])
            history["val_steps"].append(step)
            history["val_acc"].append(val_res["val_acc"])
            history["val_reward"].append(val_res["val_reward"])
            print(f"  >>> VAL @ step {step:03d}: val_acc={val_res['val_acc']*100:5.1f}% | val_r={val_res['val_reward']:.3f}")

    history["wall_sec"] = time.time() - t_start
    if cap_opt is not None:
        cap_opt.remove_hooks()
    del opts, cap_opt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return history

print("GRPO trainer ready.")


## 6. Run Apples-to-Apples Benchmark & Plot Comparison

Runs every configured arm sequentially from the exact same initial weights and prompt stream, saves `capmuon_rlvr_results.json` incrementally after each arm (safe against Colab timeouts), renders a 4-panel dark-mode comparison chart, and downloads the results.


In [ ]:
RESULTS_PATH = "capmuon_rlvr_results.json"
ALL_RESULTS = {}

for arm in CFG["ARMS"]:
    res = run_grpo_arm(model, tokenizer, arm)
    ALL_RESULTS[arm["name"]] = res
    with open(RESULTS_PATH, "w") as f:
        json.dump({"config": CFG, "results": ALL_RESULTS}, f, indent=2)
    print(f"Saved incremental results to {RESULTS_PATH} ({len(ALL_RESULTS)}/{len(CFG['ARMS'])} arms complete).")

# ==============================================================================
# SUMMARY TABLE
# ==============================================================================
print("\n" + "=" * 96)
print(f"FINAL RLVR SUMMARY — {CFG['MODEL_ID']} (GRPO mu={CFG['PPO_EPOCHS']}, {CFG['NUM_TRAIN_STEPS']} steps)")
print("=" * 96)
print(f"{'Arm':<14} {'Init Val':>9} {'Final Val':>10} {'Peak Val':>9} {'Last10 TrAcc':>13} {'Mean KL':>9} {'Mean Clip':>10} {'Time(s)':>9}")
print("-" * 96)

muon_time = ALL_RESULTS.get("muon", {}).get("wall_sec", None)
for name, res in ALL_RESULTS.items():
    init_v = res["val_acc"][0] * 100.0
    fin_v = res["val_acc"][-1] * 100.0
    peak_v = max(res["val_acc"]) * 100.0
    last10 = float(np.mean(res["train_acc"][-10:])) * 100.0 if res["train_acc"] else 0.0
    m_kl = float(np.mean(res["approx_kl"])) if res["approx_kl"] else 0.0
    m_clip = float(np.mean(res["clipfrac"])) * 100.0 if res["clipfrac"] else 0.0
    t_s = res["wall_sec"]
    ovh = f" ({(t_s / muon_time - 1)*100:+.1f}%)" if (muon_time and name.startswith("cap")) else ""
    print(f"{name:<14} {init_v:>8.1f}% {fin_v:>9.1f}% {peak_v:>8.1f}% {last10:>12.1f}% {m_kl:>9.4f} {m_clip:>9.1f}% {t_s:>8.0f}s{ovh}")

# ==============================================================================
# 4-PANEL DARK-MODE COMPARISON FIGURE
# ==============================================================================
plt.style.use("dark_background")
fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.8), facecolor="#0d1117")
colors = {
    "adamw": "#8b949e",
    "muon": "#ff6b6b",
    "cap_p025": "#4dabf7",
    "cap_p050": "#69db7c",
    "muon_3x": "#ffa94d",
    "cap_p050_3x": "#da77f2",
}


def smooth(vals, w=5):
    if len(vals) < w:
        return vals
    out = []
    for i in range(len(vals)):
        lo = max(0, i - w + 1)
        out.append(float(np.mean(vals[lo : i + 1])))
    return out


for ax in axes.flat:
    ax.set_facecolor("#0d1117")
    ax.grid(alpha=0.18, linestyle="--")

for name, res in ALL_RESULTS.items():
    c = colors.get(name, None)
    lw = 2.6 if name.startswith("cap") else 2.0
    ls = "--" if name == "adamw" else "-"

    axes[0, 0].plot(res["val_steps"], [v * 100 for v in res["val_acc"]], marker="o", lw=lw, ls=ls, color=c, label=name)
    axes[0, 1].plot(res["steps"], [v * 100 for v in smooth(res["train_acc"], 5)], lw=lw, ls=ls, color=c, label=name)
    axes[1, 0].plot(res["steps"], smooth(res["approx_kl"], 5), lw=lw, ls=ls, color=c, label=name)
    axes[1, 1].plot(res["steps"], [v * 100 for v in smooth(res["clipfrac"], 5)], lw=lw, ls=ls, color=c, label=name)

axes[0, 0].set_title("Held-Out Countdown Solve Rate (%)", fontweight="bold")
axes[0, 0].set_ylabel("Exact Solve Rate (%)")
axes[0, 0].legend(frameon=False)

axes[0, 1].set_title("Training Solve Rate (5-step MA, %)", fontweight="bold")
axes[0, 1].set_ylabel("Solve Rate (%)")

axes[1, 0].set_title(f"Multi-Epoch Policy Drift (Approx KL, μ={CFG['PPO_EPOCHS']})", fontweight="bold")
axes[1, 0].set_xlabel("GRPO Step")
axes[1, 0].set_ylabel("Unbiased k3 KL")

axes[1, 1].set_title("PPO Clip Fraction (5-step MA, %)", fontweight="bold")
axes[1, 1].set_xlabel("GRPO Step")
axes[1, 1].set_ylabel("Clip Fraction (%)")

fig.suptitle(
    f"CapMuon vs. Plain Muon vs. AdamW — LLM RLVR ({CFG['MODEL_ID']}, Countdown GRPO)",
    fontsize=13.5, fontweight="bold", y=0.99,
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
PLOT_PATH = "capmuon_rlvr_comparison.png"
plt.savefig(PLOT_PATH, dpi=180, facecolor="#0d1117")
plt.show()

# Trigger automatic file download in Google Colab
try:
    from google.colab import files
    files.download(RESULTS_PATH)
    files.download(PLOT_PATH)
    if os.path.exists("model_eigenspectrum.png"):
        files.download("model_eigenspectrum.png")
except ImportError:
    print(f"Saved {RESULTS_PATH} and {PLOT_PATH} locally.")
